In [ ]:
import pandas as pd
from typing import Dict, Any


# The file is located in the data/datasets/ directory relative to the backend folder
df = pd.read_csv('data/datasets/game_features_20260115.csv')
print(df.columns)
df.dropna(axis='columns',how='all' , inplace=True)
df.infer_objects()
df.info()
df.describe()
print(f'saving dataset, {df.shape}')
df.to_csv('data/datasets/game_features_20260115.csv')

Index(['season', 'week', 'game_id', 'home_game_date', 'home_team', 'away_team',
       'home_points_for', 'away_points_for', 'point_diff', 'winner',
       ...
       'home_vs_away_prior_games', 'home_vs_away_prior_wins',
       'home_vs_away_prior_losses', 'home_vs_away_prior_dom',
       'home_vs_away_prior_win_pct', 'away_vs_home_prior_games',
       'away_vs_home_prior_wins', 'away_vs_home_prior_losses',
       'away_vs_home_prior_dom', 'away_vs_home_prior_win_pct'],
      dtype='object', length=251)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2762 entries, 0 to 2761
Columns: 248 entries, season to away_vs_home_prior_win_pct
dtypes: float64(174), object(74)
memory usage: 5.2+ MB
saving dataset, (2762, 248)


In [17]:
import pandas as pd
import joblib
from pathlib import Path

# -----------------------------
# 1) Notebook display settings (safe)
# -----------------------------
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)

# -----------------------------
# 2) Paths (edit if needed)
# -----------------------------
PREPROCESSOR_PATH = Path(r"C:/Users/iProg/Documents/clonednfl/NFL_ML_Predictions/backend/20260115/models/preprocessor.joblib")
HOME_PIPE_PATH    = Path(r"C:/Users/iProg/Documents/clonednfl/NFL_ML_Predictions/backend/20260115/models/home_pipe.joblib")

DATASET_PATH      = Path(r"C:/Users/iProg/Documents/clonednfl/NFL_ML_Predictions/backend/data/datasets/game_features_20260115.csv")

# -----------------------------
# 3) Load artifacts (optional, but mirrors your NFL-Predict stack)
# -----------------------------
preprocessor = joblib.load(PREPROCESSOR_PATH)
home_pipe = joblib.load(HOME_PIPE_PATH)

# -----------------------------
# 4) Load dataset
# -----------------------------
df = pd.read_csv(DATASET_PATH, low_memory=False)
df.head()

for col in df.columns:
    if col.startswith('unnamed'):
        df.drop(col, axis=1, inplace=True)


for i in range(len(df)):
    if df['home_team'].values[i] == df['winner'].values[i]:
        df['winner'].values[i] = 1
    else:
        df['winner'].values[i] = 0

print(df['winner'].value_counts())
print(df.head())
df.to_csv(DATASET_PATH, index=False)

winner
1    1510
0    1252
Name: count, dtype: int64
   Unnamed: 0  season  week          game_id home_game_date home_team away_team  home_points_for  away_points_for  point_diff winner  home_prior_pf_avg_3  \
0           0  2016.0   1.0  2016_01_BUF_BAL     2016-09-11       BAL       BUF             13.0              7.0         6.0      1                  0.0   
1           1  2016.0   1.0  2016_01_CAR_DEN     2016-09-08       DEN       CAR             21.0             20.0         1.0      1                  0.0   
2           2  2016.0   1.0  2016_01_CHI_HOU     2016-09-11       HOU       CHI             23.0             14.0         9.0      1                  0.0   
3           3  2016.0   1.0  2016_01_CIN_NYJ     2016-09-11       NYJ       CIN             22.0             23.0        -1.0      0                  0.0   
4           4  2016.0   1.0  2016_01_CLE_PHI     2016-09-11       PHI       CLE             29.0             10.0        19.0      1                  0.0   

   h

In [ ]:


print("✅ Dataset loaded")
print("Shape:", df.shape)
print("Columns:", len(df.columns))

# -----------------------------
# 5) Example NFL-Predict payload
#    (this is the exact shape you described)
# -----------------------------
payload = {
    "season": 2025,
    "week": 19,
    "home_team": "JAX",
    "away_team": "BUF",
}

# -----------------------------
# 6) Helpers: normalize + row finder
# -----------------------------
TEAM_ALIAS = {
    "LA": "LAR",     # old alias → new
    "STL": "LAR",
    "SD": "LAC",
    "OAK": "LV",
    "WSH": "WAS",
}

def norm_team(team: str) -> str:
    """Normalize team abbreviations to reduce 'no match found' problems."""
    t = str(team).strip().upper()
    return TEAM_ALIAS.get(t, t)

def find_inference_rows(df: pd.DataFrame, payload: dict) -> pd.DataFrame:
    """
    Finds matching inference rows using:
      season, week, home_team, away_team

    Returns a DataFrame (0 rows = none found, >1 = duplicates exist).
    """
    season = int(payload["season"])
    week = int(payload["week"])
    home = norm_team(payload["home_team"])
    away = norm_team(payload["away_team"])

    # Normalize dataset columns on the fly (safer than trusting formatting)
    home_col = df["home_team"].astype(str).str.strip().str.upper().map(norm_team)
    away_col = df["away_team"].astype(str).str.strip().str.upper().map(norm_team)

    # Important: make sure season/week compare as numbers
    season_col = pd.to_numeric(df["season"], errors="coerce")
    week_col = pd.to_numeric(df["week"], errors="coerce")

    mask = (
        (season_col == season) &
        (week_col == week) &
        (home_col == home) &
        (away_col == away)
    )

    return df.loc[mask].copy()

# -----------------------------
# 7) Run search + print result cleanly
# -----------------------------
matches = find_inference_rows(df, payload)
print(f"\n🔎 Matches found: {len(matches)}")

if len(matches) == 0:
    print("❌ No inference row found for payload:")
    print(payload)

    # Helpful debug: show nearby candidates from same season/week
    season = int(payload["season"])
    week = int(payload["week"])
    nearby = df[(pd.to_numeric(df["season"], errors="coerce") == season) &
                (pd.to_numeric(df["week"], errors="coerce") == week)][["season", "week", "away_team", "home_team"]]
    print("\nClosest candidates in same season/week (first 15):")
    display(nearby.head(15))

elif len(matches) > 1:
    print("⚠️ Multiple rows found (duplicates). Showing key columns:")
    cols = [c for c in ["game_id", "season", "week", "away_team", "home_team", "home_game_date"] if c in matches.columns]
    display(matches[cols])

    print("\n✅ Printing FIRST match as the inference row:")
    row = matches.iloc[0]
    display(row.to_frame().T)

else:
    print("✅ Unique inference row found!\n")
    row = matches.iloc[0]

    # Print the full row as a single-row DataFrame (best readability)
    display(row.to_frame().T)

    # If you prefer raw text output too:
    # print(row.to_string())


In [ ]:

home_features = home_pipe.transform(home_features)
away_features = away_pipe.transform(away_features)
win_features = win_pipe.transform(win_features)


In [ ]:


def _build_future_row(
    df: pd.DataFrame,
    home: str,
    away: str,
    season: int,
    week: int,
) -> pd.Series:
    """
    Build a row for future games:
      - roll-forward last known team features
      - set identifiers (home/away/season/week)
      - fill remaining numeric with dataset means
    """
    # Note: ensure bundle/helpers are defined or imported if needed
    numeric_cols, categorical_cols, _ = _get_feature_columns(bundle)
    means = _dataset_means(df, numeric_cols)

    features: Dict[str, Any] = {}
    features.update(_roll_forward_team_features(df, home, season, week, "home", numeric_cols))
    features.update(_roll_forward_team_features(df, away, season, week, "away", numeric_cols))

    # Explicitly set season/week if present (fixes silent mean-fill bug)
    if "season" in numeric_cols:
        features["season"] = int(season)
    if "week" in numeric_cols:
        features["week"] = int(week)

    # Team identifiers (categorical)
    for col in categorical_cols:
        if col == "home_team":
            features[col] = home
        elif col == "away_team":
            features[col] = away
        elif col == "has_home_team":
            features[col] = True
        elif col.startswith("home_team_"):
            features[col] = (col == f"home_team_{home}")
        elif col.startswith("away_team_"):
            features[col] = (col == f"away_team_{away}")

    # Neutral defaults for common market/rest features if missing
    if "home_moneyline_prob" in numeric_cols and pd.isna(features.get("home_moneyline_prob")):
        features["home_moneyline_prob"] = means.get("home_moneyline_prob", 0.5)
    if "away_moneyline_prob" in numeric_cols and pd.isna(features.get("away_moneyline_prob")):
        features["away_moneyline_prob"] = means.get("away_moneyline_prob", 0.5)
    if "home_rest" in numeric_cols and pd.isna(features.get("home_rest")):
        features["home_rest"] = means.get("home_rest", 7.0)
    if "away_rest" in numeric_cols and pd.isna(features.get("away_rest")):
        features["away_rest"] = means.get("away_rest", 7.0)

    # Simple derived diffs
    if "moneyline_prob_diff" in numeric_cols:
        h, a = features.get("home_moneyline_prob"), features.get("away_moneyline_prob")
        if pd.notna(h) and pd.notna(a):
            features["moneyline_prob_diff"] = float(h) - float(a)

    if "rest_diff" in numeric_cols:
        h, a = features.get("home_rest"), features.get("away_rest")
        if pd.notna(h) and pd.notna(a):
            features["rest_diff"] = float(h) - float(a)

    if "elo_diff_pre" in numeric_cols:
        h, a = features.get("home_elo_pre"), features.get("away_elo_pre")
        if pd.notna(h) and pd.notna(a):
            features["elo_diff_pre"] = float(h) - float(a)

    # General “home_minus_away_*” features
    for col in numeric_cols:
        if not col.startswith("home_minus_away_"):
            continue
        suffix = col[len("home_minus_away_"):]
        h_col = f"home_{suffix}"
        a_col = f"away_{suffix}"
        h, a = features.get(h_col), features.get(a_col)
        if pd.notna(h) and pd.notna(a):
            features[col] = float(h) - float(a)

    # Fill remaining numeric gaps
    for col in numeric_cols:
        if col not in features or pd.isna(features.get(col)):
            features[col] = means.get(col, 0.0)

    return pd.Series(features)


# row = _build_future_row(df, home, away, season, week)
# print(row)
